# 03 — Triển khai mô hình SVM và So sánh với Random Forest

**Mục tiêu**: Đào tạo mô hình Support Vector Machine (LinearSVC) trên tập dữ liệu OULAD, sau đó so sánh trực diện (1-vs-1) với mô hình Random Forest về cả độ chính xác lẫn tốc độ chạy.

**Quy trình**:
1. Load dữ liệu (`X.parquet`, `y.parquet`).
2. Xây dựng 2 Pipeline riêng biệt (SVM dùng `StandardScaler`, RF không cần).
3. Đánh giá Baseline cả 2 mô hình với 5-fold CV và xuất Bảng So sánh.
4. Tìm kiếm siêu tham số tối ưu (Hyperparameter Tuning) cho SVM.
5. Lưu mô hình tốt nhất để tích hợp Backend.

In [ ]:
# 1. Import thư viện
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path('..').resolve()
DATA_PROCESSED = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. Load dữ liệu
print("Đang nạp dữ liệu...")
X = pd.read_parquet(DATA_PROCESSED / 'X.parquet')
y = pd.read_parquet(DATA_PROCESSED / 'y.parquet')['target']

with open(DATA_PROCESSED / 'feature_metadata.json', 'r', encoding='utf-8') as f:
    META = json.load(f)

print(f'Số lượng mẫu (Rows) = {X.shape[0]:,}')
print(f'Số lượng đặc trưng (Features) = {X.shape[1]}')

In [ ]:
# 3. Xây dựng Pipeline cho SVM và Random Forest
numeric_cols = META["numeric_cols"]
categorical_ordinal = META["categorical_ordinal"]
categorical_nominal = META["categorical_nominal"]
ordinal_categories = META["ordinal_categories"]

# --- PREPROCESSOR CHO SVM (CÓ SCALING) ---
preprocessor_svm = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_cols),
    ("ord", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encode", OrdinalEncoder(
            categories=[ordinal_categories[c] + ["Unknown"] for c in categorical_ordinal],
            handle_unknown="use_encoded_value", unknown_value=-1)),
        ("scale", StandardScaler())
    ]), categorical_ordinal),
    ("nom", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_nominal),
])

# --- PREPROCESSOR CHO RF (KHÔNG SCALING) ---
preprocessor_rf = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),
    ("ord", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encode", OrdinalEncoder(
            categories=[ordinal_categories[c] + ["Unknown"] for c in categorical_ordinal],
            handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), categorical_ordinal),
    ("nom", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_nominal),
])

pipe_svm = Pipeline([("prep", preprocessor_svm), ("svm", LinearSVC(random_state=42, dual=False, C=1.0))])
pipe_rf = Pipeline([("prep", preprocessor_rf), ("rf", RandomForestClassifier(random_state=42, n_jobs=-1, n_estimators=100))])

In [ ]:
# 4. Trận đấu (Head-to-Head Comparison)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = ['accuracy', 'f1']

def evaluate_model(pipe, name: str) -> dict:
    t0 = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1)
    fit_time = time.time() - t0
    return {
        'Model': name,
        'Test Accuracy': cv_res['test_accuracy'].mean(),
        'Train Accuracy': cv_res['train_accuracy'].mean(),
        'Overfit Gap': cv_res['train_accuracy'].mean() - cv_res['test_accuracy'].mean(),
        'F1-Score': cv_res['test_f1'].mean(),
        'Total Time (s)': fit_time
    }

print("Đang chạy 5-Fold CV cho LinearSVC...")
res_svm = evaluate_model(pipe_svm, "LinearSVC (Baseline)")

print("Đang chạy 5-Fold CV cho Random Forest...")
res_rf = evaluate_model(pipe_rf, "Random Forest (Baseline)")

# Tạo bảng so sánh
df_compare = pd.DataFrame([res_svm, res_rf])
df_compare.set_index('Model', inplace=True)

print("\n" + "="*50)
print("BẢNG SO SÁNH HIỆU SUẤT: LINEARSVC vs RANDOM FOREST")
print("="*50)
print(df_compare.round(4).to_string())
print("="*50)

In [ ]:
# 5. Grid Search tối ưu hóa tham số C cho SVM
print("\nĐang chạy GridSearch để tìm tham số C tối ưu cho LinearSVC...")
param_grid = {'svm__C': [0.001, 0.01, 0.1, 1.0, 10.0]}

grid_search = GridSearchCV(
    pipe_svm, param_grid, cv=CV, scoring='accuracy', n_jobs=-1, return_train_score=True
)

t0 = time.time()
grid_search.fit(X, y)
print(f"GridSearch hoàn tất trong {time.time() - t0:.1f} giây")

best_C = grid_search.best_params_['svm__C']
print(f"Tham số tốt nhất: C = {best_C}")
print(f"Độ chính xác tương ứng: {grid_search.best_score_:.4f}")

In [ ]:
# 6. Trực quan hóa kết quả Grid Search
results = pd.DataFrame(grid_search.cv_results_)

plt.figure(figsize=(8, 5))
plt.plot(results['param_svm__C'].astype(float), results['mean_train_score'], label='Train Accuracy', marker='o')
plt.plot(results['param_svm__C'].astype(float), results['mean_test_score'], label='Test (CV) Accuracy', marker='s')
plt.xscale('log')
plt.xlabel('Tham số C (log scale)')
plt.ylabel('Độ chính xác (Accuracy)')
plt.title('Ảnh hưởng của tham số C đến độ chính xác LinearSVC')
plt.legend()
plt.grid(True)
plt.savefig(FIG_DIR / 'svm_c_tuning.png')
plt.show()

In [ ]:
# 7. Lưu Best Parameters và Best Model
best_svm_params = {'C': best_C}

with open(DATA_PROCESSED / 'best_svm_params.json', 'w', encoding='utf-8') as f:
    json.dump(best_svm_params, f, indent=2)

print("Đã lưu tham số tối ưu vào data/processed/best_svm_params.json")

best_model = grid_search.best_estimator_
joblib.dump(best_model, MODELS_DIR / 'best_svm_pipeline.joblib')
print("Đã lưu mô hình (Pipeline) hoàn chỉnh vào models/best_svm_pipeline.joblib")